In [1]:
!pip install huggingface_hub datasets pandas

In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [5]:
import os
import glob
import json
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from google.colab import drive

# 1. Mount Drive so this new notebook can read your preprocessed data
drive.mount('/content/drive')

# --- CONFIGURATION ---
PROJECT_BASE = "/content/drive/MyDrive/TRIBE_Preprocessing/outputs/HCP"
# ⚠️ Update this string to your target Hugging Face username and repo choice
HF_REPO_ID   = "archi0918/hcp-preprocessed-fmri"

all_records = []

# Find all metadata json logs inside your Drive structure
metadata_files = glob.glob(os.path.join(PROJECT_BASE, "sub-*", "*_metadata.json"))

# FIXED: Extract the digits from the folder name "sub-XYZ" to sort numerically!
def get_subject_number(filepath):
    # This grabs the folder name (e.g., 'sub-10') and extracts just the number (10)
    folder_name = filepath.split(os.sep)[-2]
    num_part = folder_name.replace("sub-", "")
    return int(num_part)

# Sort using our custom numerical key function
sorted_metadata_files = sorted(metadata_files, key=get_subject_number)

print(f"Found {len(sorted_metadata_files)} session records to compile.")

for meta_path in sorted_metadata_files:
    with open(meta_path, 'r') as f:
        meta_data = json.load(f)

    subject_dir = os.path.dirname(meta_path)
    filename = os.path.basename(meta_path)

    # Dynamically extract the full run details between the subject name and the suffix
    # e.g., "sub-174_task-pixar_space-MNI..._metadata.json" -> "task-pixar_space-MNI..."
    subject_prefix = f"sub-{meta_data['subject']}_"
    run_name = filename.replace(subject_prefix, "").replace("_metadata.json", "")

    # Derive corresponding matrix array paths
    cortex_file = os.path.join(subject_dir, f"{subject_prefix}{run_name}_cortical_parcels.npy")
    subcortex_file = os.path.join(subject_dir, f"{subject_prefix}{run_name}_subcortical_parcels.npy")

    if os.path.exists(cortex_file) and os.path.exists(subcortex_file):
        # Load matrices and convert to list-of-lists for Hugging Face compatibility
        cortex_matrix = np.load(cortex_file).tolist()
        subcortex_matrix = np.load(subcortex_file).tolist()

        # Flatten out the row schema matching your target preview layout
        row_entry = {
            "subject": f"sub-{meta_data['subject']}",
            "dataset": meta_data["dataset"].lower(),
            "run": run_name,
            "native_tr_seconds": float(meta_data["native_tr_seconds"]),
            "resampled_hz": int(meta_data["resampled_hz"]),
            "hemodynamic_lag_seconds": 5,
            "shapes": {
                "cortical_parcels": [len(cortex_matrix), len(cortex_matrix[0])],
                "subcortical_parcels": [len(subcortex_matrix), len(subcortex_matrix[0])]
            },
            "cortical_parcels": cortex_matrix,
            "subcortical_parcels": subcortex_matrix
        }
        all_records.append(row_entry)

# 4. Construct the Hugging Face Dataset directly from the tabular list
if all_records:
    print("\nFormatting database tables...")
    df = pd.DataFrame(all_records)
    hf_dataset = Dataset.from_pandas(df)

    # Wrap it inside a standard train split configuration
    dataset_dict = DatasetDict({"train": hf_dataset})

    print(f"Pushing pipeline tables to Hugging Face Hub: {HF_REPO_ID}...")
    dataset_dict.push_to_hub(HF_REPO_ID, private=False)
    print("\n[SUCCESS] Upload finished! Check your Hugging Face page to verify the tabular preview layout.")
else:
    print("!! Error: No complete matching metadata/matrix vector outputs found in Drive path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 177 session records to compile.

Formatting database tables...
Pushing pipeline tables to Hugging Face Hub: archi0918/hcp-preprocessed-fmri...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   9%|8         | 15.7MB /  183MB            

README.md:   0%|          | 0.00/731 [00:00<?, ?B/s]


[SUCCESS] Upload finished! Check your Hugging Face page to verify the tabular preview layout.
